In [9]:
import pandas as pd
import numpy as np

df = pd.read_excel('../data/Telco_customer_churn.xlsx')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')
df = df.dropna(subset=['total_charges'])

In [10]:
categorical_cols = [
    'gender','senior_citizen','partner','dependents',
    'phone_service','multiple_lines','internet_service',
    'online_security','online_backup','device_protection',
    'tech_support','streaming_tv','streaming_movies',
    'contract','paperless_billing','payment_method'
]

numeric_cols = [
    'tenure_months',
    'monthly_charges',
    'total_charges'
]

X = df[categorical_cols + numeric_cols]
y = df['churn_value']

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['gender', 'senior_citizen',
                                                   'partner', 'dependents',
                                                   'phone_service',
                                                   'multiple_lines',
                                                   'internet_service',
                                                   'online_security',
                                                   'online_backup',
                                                   'device_protection',
                                                   'tech_support',
                                                   'streaming_tv',
                                                   'streaming_movies',
                                          

In [14]:
from sklearn.metrics import classification_report

# class_weight 적용
rf_model_balanced = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced'
    ))
])

rf_model_balanced.fit(X_train, y_train)

y_pred_balanced = rf_model_balanced.predict(X_test)

print("=== Balanced Model Performance ===")
print(classification_report(y_test, y_pred_balanced))



=== Balanced Model Performance ===
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.63      0.51      0.57       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407

